# Kaggle ML Lab V4
Model contracts, dataset compatibility, benchmarking and FastAPI local simulation.


In [ ]:
# KAGGLE ML LAB V4 — MODEL CONTRACTS, BENCHMARKING, FASTAPI & TYPED INPUT
# One cell. Attach trusted .pt/.joblib models and Kaggle Dataset(s) via Add Input.
# No pip installation, network calls or automatic benchmarking.
import json, time, hashlib
from pathlib import Path
import numpy as np, pandas as pd, joblib, torch
from torch import nn
import ipywidgets as w
from IPython.display import display, clear_output
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score)
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

ROOT = Path('/kaggle/input')
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUT.mkdir(parents=True, exist_ok=True)
LOADED = {}
DATASETS = {}
# Optional overrides keyed by model id or model artifact filename, for models missing metadata.
# Example: CONTRACT_OVERRIDES['your_model.joblib'] = {'target':'churned','task':'classification'}
CONTRACT_OVERRIDES = {}
KNOWN_TARGETS = ('delay_minutes','target','label','churned','y','price','sales')

class RebuiltMLP(nn.Module):
    # Compatible only with checkpoints whose architecture is Linear -> ReLU -> ... -> Linear.
    def __init__(self, dims):
        super().__init__()
        layers=[]
        for i in range(len(dims)-1):
            layers.append(nn.Linear(dims[i],dims[i+1]))
            if i<len(dims)-2: layers.append(nn.ReLU())
        self.network=nn.Sequential(*layers)
    def forward(self,x): return self.network(x)

def model_paths():
    if not ROOT.exists(): return []
    found=[p for p in ROOT.rglob('*') if p.is_file() and p.suffix.lower() in ('.pt','.joblib')
           and not any(s in p.stem.lower() for s in ('preprocessor','scaler','encoder','imputer','tokenizer'))]
    # Remove model_factory / model_factory_artifacts aliases in same version.
    unique={}
    for p in sorted(found,key=lambda p: ('model_factory_artifacts' not in p.parts,str(p))):
        key=str(p).replace('/model_factory_artifacts/','/MODEL_COPY/').replace('/model_factory/','/MODEL_COPY/')
        unique.setdefault(key,p)
    return list(unique.values())

def dataset_packages():
    result={}
    if not ROOT.exists(): return result
    for p in sorted(ROOT.rglob('*.csv')):
        if '/models/' in str(p).lower(): continue
        rel=p.relative_to(ROOT).parts
        if rel[0]=='datasets' and len(rel)>=4:
            folder=ROOT.joinpath(*rel[:3])
        else: folder=ROOT/rel[0]
        result.setdefault(str(folder),[]).append(p)
    return result

def choose_csv(paths):
    return sorted(paths,key=lambda p:(0 if any(z in p.stem.lower() for z in ('test','eval','valid','holdout')) else 1,str(p)))[0]

def read_model(path):
    p=Path(path)
    if p.suffix.lower()=='.joblib':
        obj=joblib.load(p)  # ONLY trusted artifacts: pickle/joblib can execute code.
        if not callable(getattr(obj,'predict',None)):
            raise ValueError('Artifact is not a predictive estimator')
        return obj,'estimator',{}
    try:
        obj=torch.jit.load(str(p),map_location='cpu'); obj.eval()
        return obj,'torchscript',{}
    except Exception: pass
    ckpt=torch.load(str(p),map_location='cpu',weights_only=True)
    if not isinstance(ckpt,dict) or 'state_dict' not in ckpt or 'dimensions' not in ckpt:
        raise ValueError('Unsupported PyTorch checkpoint format')
    dims=[int(z) for z in ckpt['dimensions']]
    weights=[(k,v) for k,v in ckpt['state_dict'].items() if k.endswith('.weight') and isinstance(v,torch.Tensor) and v.ndim==2]
    if len(weights)!=len(dims)-1: raise ValueError('Not a Linear/ReLU MLP checkpoint')
    obj=RebuiltMLP(dims)
    linears=[m for m in obj.network if isinstance(m,nn.Linear)]
    with torch.no_grad():
        for layer,(key,value) in zip(linears,weights):
            bias=ckpt['state_dict'].get(key[:-7]+'.bias')
            if bias is None or tuple(layer.weight.shape)!=tuple(value.shape) or tuple(layer.bias.shape)!=tuple(bias.shape):
                raise ValueError('Checkpoint layer weights incompatible: '+key)
            layer.weight.copy_(value); layer.bias.copy_(bias)
    obj.eval()
    return obj,'torch_checkpoint',ckpt

def preprocessor_for(path):
    # Scoped to this model version; cannot silently borrow another model's preprocessing.
    p=Path(path)
    directory=p.parent.parent
    found=sorted(x for x in directory.rglob('*.joblib') if 'preprocessor' in x.stem.lower())
    if not found: return None,None
    return joblib.load(found[0]),str(found[0])

def normal_task(raw):
    s=str(raw or '').lower().strip()
    if s in ('regression','regressor','reg'): return 'regression'
    if s in ('classification','classifier','binary','multiclass','class'): return 'classification'
    return None

def contract_for(name,path,obj,kind,ckpt,prep):
    override={**CONTRACT_OVERRIDES.get(Path(path).name,{}),**CONTRACT_OVERRIDES.get(name,{})}
    training_features=list(getattr(prep,'feature_names_in_',[])) if prep is not None else []
    if not training_features: training_features=list(getattr(obj,'feature_names_in_',[]))
    task=(override.get('task') or normal_task(ckpt.get('task')) or
          ('classification' if getattr(obj,'_estimator_type',None)=='classifier' else
           'regression' if getattr(obj,'_estimator_type',None)=='regressor' else None))
    target=override.get('target') or ckpt.get('target') or ckpt.get('label_column')
    # No invented target mapping: a model with unknown target must be confirmed by user.
    return {'features':training_features,'task':task,'target':target,
            'output_mode':override.get('output_mode',ckpt.get('output_mode','logit')),
            'architecture_unverified':kind=='torch_checkpoint'}

def add_overrides(name,task=None,target=None):
    if name not in LOADED: return
    c=LOADED[name]['contract']
    if task: c['task']=task
    if target: c['target']=target

def sanitize(X):
    X=X.copy()
    for col in X.select_dtypes(include=['object','string']).columns:
        X[col]=X[col].fillna('').astype(str)
    return X

def infer(name,X):
    item=LOADED[name]; c=item['contract']; obj=item['model']
    expected=c['features']
    if expected:
        missing=[z for z in expected if z not in X.columns]
        if missing: raise ValueError('Missing required model INPUT features: '+str(missing))
        X=X.loc[:,expected]
    X=sanitize(X)
    if item['kind']=='estimator': return np.asarray(obj.predict(X)).reshape(-1)
    if item['prep'] is not None: X=item['prep'].transform(X)
    if hasattr(X,'toarray'): X=X.toarray()
    vals=X.to_numpy(dtype=np.float32) if hasattr(X,'to_numpy') else np.asarray(X,dtype=np.float32)
    if vals.ndim!=2 or not np.isfinite(vals).all():
        raise ValueError('Prepared features must be finite numeric values')
    if item['kind']=='torch_checkpoint' and vals.shape[1]!=int(item['ckpt']['dimensions'][0]):
        raise ValueError('Expected '+str(item['ckpt']['dimensions'][0])+' transformed features, received '+str(vals.shape[1]))
    with torch.no_grad():
        raw=obj(torch.as_tensor(vals,dtype=torch.float32))
        if isinstance(raw,(tuple,list)): raw=raw[0]
        raw=raw.detach().cpu().numpy()
    task=c['task']
    if task=='regression': return np.asarray(raw).reshape(-1)
    if task=='classification':
        if raw.ndim==2 and raw.shape[1]>1: return raw.argmax(axis=1)
        threshold=.5 if c['output_mode']=='probability' else 0.0
        return (raw.reshape(-1)>=threshold).astype(int)
    raise ValueError('Model task is unknown; configure its contract before inference')

# Only ONE set of UI controls. No widget duplication inside refresh callbacks.
model_ui=w.SelectMultiple(description='Models',rows=6,layout=w.Layout(width='98%'))
dataset_ui=w.Dropdown(description='Dataset',layout=w.Layout(width='98%'))
target_ui=w.Dropdown(description='Target',layout=w.Layout(width='65%'))
task_ui=w.Dropdown(description='Task',options=['regression','classification'])
model_contract_ui=w.Dropdown(description='Contract model',layout=w.Layout(width='95%'))
confirm_btn=w.Button(description='Confirm model contract')
refresh_btn=w.Button(description='Refresh inputs')
load_btn=w.Button(description='Load models',button_style='success')
check_btn=w.Button(description='Check compatibility')
bench_btn=w.Button(description='Benchmark',button_style='primary')
query_btn=w.Button(description='Query API')
manual_model_ui=w.Dropdown(description='Serve',layout=w.Layout(width='95%'))
manual_input=w.Textarea(description='Typed JSON',value='{}',layout=w.Layout(width='98%',height='180px'))
manual_btn=w.Button(description='Predict typed JSON',button_style='success')
log=w.Output(); manual_log=w.Output()
SELECTED_CSV=None

def update_dataset(change=None):
    global SELECTED_CSV
    paths=DATASETS.get(dataset_ui.value,[])
    SELECTED_CSV=choose_csv(paths) if paths else None
    cols=pd.read_csv(SELECTED_CSV,nrows=1).columns.tolist() if SELECTED_CSV else []
    old=target_ui.value
    target_ui.options=cols
    target_ui.value=old if old in cols else next((k for k in KNOWN_TARGETS if k in cols),cols[-1] if cols else None)
    if target_ui.value=='delay_minutes': task_ui.value='regression'

def refresh(_=None):
    global DATASETS
    old_models=set(model_ui.value); old_dataset=dataset_ui.value
    paths=model_paths(); DATASETS=dataset_packages()
    model_ui.options=[(str(p),str(p)) for p in paths]
    model_ui.value=tuple(str(p) for p in paths if str(p) in old_models)
    dataset_ui.options=[(p,p) for p in sorted(DATASETS)]
    if old_dataset in DATASETS: dataset_ui.value=old_dataset
    update_dataset()
    with log:
        clear_output(wait=True)
        print(f'Available: {len(paths)} model files, {len(DATASETS)} dataset attachments')
        print('Selected dataset file:',SELECTED_CSV,'| target:',target_ui.value)

def load(_=None):
    LOADED.clear()
    with log:
        clear_output(wait=True)
        for path in model_ui.value:
            try:
                obj,kind,ckpt=read_model(path)
                prep,prep_path=preprocessor_for(path) if kind!='estimator' else (None,None)
                # Stable unique model name by Kaggle package slug.
                parts=Path(path).parts
                package=parts[parts.index('models')+2] if 'models' in parts and len(parts)>parts.index('models')+2 else Path(path).parent.name
                name=package+'_'+Path(path).stem
                count=2; base=name
                while name in LOADED: name=base+'_'+str(count); count+=1
                c=contract_for(name,path,obj,kind,ckpt,prep)
                LOADED[name]={'model':obj,'kind':kind,'ckpt':ckpt,'prep':prep,'path':path,'contract':c}
                print('LOADED',name,'|',kind,'| task:',c['task'],'| target:',c['target'])
                print('   Required inputs:',c['features'],'| preprocessor:',prep_path)
                if c['architecture_unverified']: print('   NOTE: Linear/ReLU assumption; verify architecture and outputs.')
            except Exception as exc: print('FAILED:',path,'|',exc)
        model_contract_ui.options=list(LOADED)
        manual_model_ui.options=[('All loaded','__all__')]+[(n,n) for n in LOADED]
        print('Inference models:',len(LOADED))

def confirm_contract(_=None):
    name=model_contract_ui.value
    if not name: return
    add_overrides(name,task_ui.value,target_ui.value)
    with log:
        clear_output(wait=True)
        print('Confirmed:',name,'=>',LOADED[name]['contract'])
        print('Confirm only the target and task used when the model was TRAINED.')

def dataset():
    if SELECTED_CSV is None: raise ValueError('Select an attached Kaggle Dataset first')
    frame=pd.read_csv(SELECTED_CSV)
    target=target_ui.value
    if target not in frame: raise ValueError('Selected target not present in dataset')
    frame=frame.dropna(subset=[target])
    if frame.empty: raise ValueError('No rows with labels')
    return sanitize(frame.drop(columns=[target])),frame[target]

def validate_pair(name,X,target,task):
    c=LOADED[name]['contract']
    if c['target'] is None: raise ValueError('Unknown training target; confirm contract before benchmarking')
    if c['task'] is None: raise ValueError('Unknown training task; confirm contract before benchmarking')
    if c['target']!=target: raise ValueError(f'Target mismatch: trained for {c["target"]}, selected {target}')
    if c['task']!=task: raise ValueError(f'Task mismatch: trained for {c["task"]}, selected {task}')
    if not c['features']: raise ValueError('Missing feature schema. Supply original model preprocessing/metadata.')
    missing=[k for k in c['features'] if k not in X.columns]
    if missing: raise ValueError('Missing required input features: '+str(missing))
    if target in c['features']: raise ValueError('Selected target is a required model input, not its prediction label')

def check(_=None):
    with log:
        clear_output(wait=True)
        try:
            X,y=dataset(); print('Dataset:',SELECTED_CSV,'| target:',target_ui.value,'| rows:',len(X))
            for name in LOADED:
                try:
                    validate_pair(name,X,target_ui.value,task_ui.value)
                    print('PASS:',name,'=>',infer(name,X.iloc[:1]).tolist())
                except Exception as exc: print('INCOMPATIBLE:',name,'=>',exc)
        except Exception as exc: print('CHECK ERROR:',exc)

def benchmark():
    X,y=dataset()
    if not LOADED: raise ValueError('Load models first')
    records=[]
    for name,item in LOADED.items():
        try:
            validate_pair(name,X,target_ui.value,task_ui.value)
            infer(name,X.iloc[:min(3,len(X))])
            timings=[]
            for _ in range(7):
                begin=time.perf_counter(); pred=infer(name,X)
                timings.append((time.perf_counter()-begin)*1000)
            row={'model':name,'status':'OK','rows':len(X),'batch_p50_ms':float(np.median(timings)),
                 'batch_p95_ms':float(np.percentile(timings,95)),
                 'rows_per_second':float(len(X)*1000/np.mean(timings)),
                 'artifact_mb':Path(item['path']).stat().st_size/1048576}
            if task_ui.value=='regression':
                actual=pd.to_numeric(y,errors='raise').to_numpy(dtype=float)
                predicted=np.asarray(pred,dtype=float)
                row.update(mae=float(mean_absolute_error(actual,predicted)),
                           rmse=float(np.sqrt(mean_squared_error(actual,predicted))),r2=float(r2_score(actual,predicted)))
            else:
                row.update(accuracy=float(accuracy_score(y,pred)),
                           precision=float(precision_score(y,pred,average='weighted',zero_division=0)),
                           recall=float(recall_score(y,pred,average='weighted',zero_division=0)),
                           f1=float(f1_score(y,pred,average='weighted',zero_division=0)))
            records.append(row)
        except Exception as exc: records.append({'model':name,'status':'SKIPPED','reason':str(exc)})
    board=pd.DataFrame(records)
    metric='rmse' if task_ui.value=='regression' else 'f1'
    if metric in board: board=board.sort_values(metric,ascending=(metric=='rmse'),na_position='last')
    board.to_csv(OUT/'benchmark_leaderboard.csv',index=False)
    return board

app=FastAPI(title='Kaggle Model Serving Lab')
class Request(BaseModel):
    instances:list[dict]
@app.get('/health')
def health(): return {'ok':True,'models_loaded':len(LOADED)}
@app.get('/models')
def get_models(): return [{'name':k,'contract':v['contract']} for k,v in LOADED.items()]
@app.post('/predict/{name}')
def api_one(name:str,request:Request):
    if name not in LOADED: raise HTTPException(404,'Unknown model')
    try: return {'model':name,'predictions':infer(name,pd.DataFrame(request.instances)).tolist()}
    except Exception as exc: raise HTTPException(422,str(exc))
@app.post('/predict-all')
def api_all(request:Request):
    X=pd.DataFrame(request.instances); result={}
    for name in LOADED:
        try: result[name]=infer(name,X).tolist()
        except Exception as exc: result[name]={'error':str(exc)}
    return {'results':result}
client=TestClient(app)

def run_benchmark(_=None):
    with log:
        clear_output(wait=True)
        try: display(benchmark()); print('Saved:',OUT/'benchmark_leaderboard.csv')
        except Exception as exc: print('BENCHMARK ERROR:',exc)

def query(_=None):
    with log:
        clear_output(wait=True)
        try:
            X,_=dataset(); request={'instances':json.loads(X.iloc[:1].to_json(orient='records'))}
            response=client.post('/predict-all',json=request)
            print('HTTP',response.status_code, json.dumps(response.json(),indent=2))
        except Exception as exc: print('API ERROR:',exc)

def typed_predict(_=None):
    with manual_log:
        clear_output(wait=True)
        try:
            data=json.loads(manual_input.value)
            instances=data['instances'] if isinstance(data,dict) and 'instances' in data else data if isinstance(data,list) else [data]
            if not isinstance(instances,list) or not instances or any(not isinstance(z,dict) for z in instances):
                raise ValueError('Enter a JSON object, array of objects, or {"instances":[...]}')
            name=manual_model_ui.value
            endpoint='/predict-all' if name=='__all__' else '/predict/'+name
            start=time.perf_counter(); response=client.post(endpoint,json={'instances':instances})
            print('POST',endpoint,'HTTP',response.status_code,'round_trip_ms',round((time.perf_counter()-start)*1000,2))
            print(json.dumps(response.json(),indent=2))
        except Exception as exc: print('INPUT ERROR:',exc)

dataset_ui.observe(update_dataset,names='value')
refresh_btn.on_click(refresh); load_btn.on_click(load); confirm_btn.on_click(confirm_contract)
check_btn.on_click(check); bench_btn.on_click(run_benchmark); query_btn.on_click(query)
manual_btn.on_click(typed_predict)
display(w.HTML('<h3>ML Lab V4 — model contracts & compatible benchmarks</h3>'))
display(model_ui,dataset_ui,target_ui,task_ui)
display(w.HBox([refresh_btn,load_btn,check_btn,bench_btn,query_btn]))
display(w.HBox([model_contract_ui,confirm_btn]),log)
display(w.HTML('<h3>FastAPI typed prediction console</h3>'))
display(manual_model_ui,manual_input,manual_btn,manual_log)
refresh()
print('READY. Load models; confirm each model trained task/target; check; benchmark; type JSON to serve.')
